In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, precision_recall_curve, recall_score
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# (Setup device, path, dan seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

DATA_PATH = r'C:\FRAUD-Sinta2\Output'
MODEL_PATH = r'C:\FRAUD-Sinta2\Output\Olist_Experiment'
os.makedirs(MODEL_PATH, exist_ok=True)
RANDOM_SEED = 42
TIME_STEPS = 10 # (PENTING: Jaga agar sama dengan N2)
BATCH_SIZE = 128
N_EPOCHS = 30 # (AE Statis)
N_EPOCHS_LSTM = 40 # (Kita latih LSTM lebih lama di sini)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("Libraries imported and setup complete.")


Menggunakan device: cuda
Libraries imported and setup complete.


In [2]:
try:
    df = pd.read_csv(os.path.join(DATA_PATH, 'simulasi_data_final_v2.csv'))
    print(f"Dataset Simulasi V2 berhasil di-load. Shape: {df.shape}")
except FileNotFoundError:
    print("Error: Pastikan file 'simulasi_data_final_v2.csv' ada.")

# --- Preprocessing Khas Data Olist ---

# 1. Hapus kolom 'order_id' (tidak relevan untuk model)
df.drop('order_id', axis=1, inplace=True, errors='ignore')

# 2. Konversi 'timestamp' ke fitur numerik
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour_of_day'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
# (Kita tidak bisa sort berdasarkan 'Time' karena data sudah di-acak saat pembuatan)
# (Kita akan sort berdasarkan user_id dan timestamp untuk data sekuensial)
df.sort_values(by=['user_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

# 3. Label Encode 'user_id' (string -> angka)
le = LabelEncoder()
df['user_id_encoded'] = le.fit_transform(df['user_id'])

# 4. Scaling 'amount'
scaler = StandardScaler()
df['amount_scaled'] = scaler.fit_transform(df[['amount']])

# 5. Buat DataFrame Final untuk Model
# (Kita simpan 'anomaly_type' dan 'Class' untuk label)
labels_df = df[['Class', 'anomaly_type']]
# (Fitur yang akan digunakan model)
features_df = df.drop(
    ['user_id', 'timestamp', 'amount', 'Class', 'anomaly_type'], 
    axis=1
)
print("Preprocessing Olist selesai. Fitur yang digunakan:", features_df.columns.tolist())

# --- Split Data Kronologis (berdasarkan index setelah sort) ---
n = len(features_df)
train_size = int(n * 0.70)
val_size = int(n * 0.15)
test_size = n - train_size - val_size 

# 0% --- 70%
df_train_full = features_df.iloc[:train_size]
df_train_full['Class'] = labels_df.iloc[:train_size]['Class']

# 70% --- 85%
df_val_full = features_df.iloc[train_size:train_size+val_size]
df_val_full['Class'] = labels_df.iloc[train_size:train_size+val_size]['Class']

# 85% --- 100%
df_test_mixed = features_df.iloc[train_size+val_size:]
df_test_mixed['Class'] = labels_df.iloc[train_size+val_size:]['Class']
df_test_mixed['anomaly_type'] = labels_df.iloc[train_size+val_size:]['anomaly_type'] # (PENTING untuk Tabel 4)

# Pisahkan data normal untuk melatih AE
df_train_normal = df_train_full[df_train_full['Class'] == 0].copy()

# Pisahkan X dan y untuk baseline supervised (SMOTE)
X_train_statis = df_train_full.drop('Class', axis=1)
y_train_statis = df_train_full['Class']
X_test_statis = df_test_mixed.drop(['Class', 'anomaly_type'], axis=1)
y_test_statis = df_test_mixed['Class']

print(f"Data Normal untuk Training AE: {df_train_normal.shape}")
print(f"Data Full untuk Validasi: {df_val_full.shape}")
print(f"Data Full untuk Test: {df_test_mixed.shape}")


Dataset Simulasi V2 berhasil di-load. Shape: (105372, 6)
Preprocessing Olist selesai. Fitur yang digunakan: ['hour_of_day', 'day_of_week', 'month', 'user_id_encoded', 'amount_scaled']
Data Normal untuk Training AE: (72335, 6)
Data Full untuk Validasi: (15805, 6)
Data Full untuk Test: (15807, 7)


In [3]:
print("\n--- [Baseline 1] Melatih Isolation Forest (Simulasi) ---")

contamination_rate = labels_df['Class'].value_counts(normalize=True)[1]
print(f"Contamination rate (simulasi): {contamination_rate:.5f}")

model_if = IsolationForest(
    n_estimators=100,
    contamination=contamination_rate,
    max_samples='auto',
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print("Melatih IF pada data training normal...")
model_if.fit(df_train_normal.drop('Class', axis=1))

print("Mengevaluasi IF pada data tes...")
y_pred_if_raw = model_if.predict(X_test_statis)
y_pred_if = [1 if p == -1 else 0 for p in y_pred_if_raw]
y_true_if = y_test_statis

print("\n--- Hasil Isolation Forest (Tabel 2) ---")
print(classification_report(y_true_if, y_pred_if, target_names=['Normal (0)', 'Fraud (1)']))



--- [Baseline 1] Melatih Isolation Forest (Simulasi) ---
Contamination rate (simulasi): 0.01993
Melatih IF pada data training normal...
Mengevaluasi IF pada data tes...

--- Hasil Isolation Forest (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       1.00      0.96      0.98     15480
   Fraud (1)       0.30      0.89      0.45       327

    accuracy                           0.96     15807
   macro avg       0.65      0.92      0.71     15807
weighted avg       0.98      0.96      0.97     15807



In [4]:
print("\n--- [Baseline 2] Melatih Supervised (XGBoost) + SMOTE (Simulasi) ---")

n_fraud_train = y_train_statis.sum()
print(f"Jumlah fraud di set training (70% pertama): {n_fraud_train}")

if n_fraud_train < 6:
    print("Error: Fraud di set training terlalu sedikit untuk k_neighbors=5. Melewatkan baseline SMOTE.")
else:
    smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
    print("Menerapkan SMOTE pada data training...")
    
    X_train_smote, y_train_smote = smote.fit_resample(X_train_statis, y_train_statis)
    print(f"Shape data training SMOTE: {X_train_smote.shape}")

    print("Melatih XGBClassifier pada data SMOTE...")
    model_xgb_smote = XGBClassifier(
        random_state=RANDOM_SEED, 
        n_estimators=100, 
        eval_metric='logloss',
        n_jobs=-1,
        tree_method='hist', 
        device=device.type 
    )
    model_xgb_smote.fit(X_train_smote, y_train_smote)

    print("Mengevaluasi model pada data tes kronologis asli...")
    y_pred_xgb_smote = model_xgb_smote.predict(X_test_statis)
    y_true_xgb = y_test_statis

    print("\n--- Hasil XGBoost + SMOTE (Tabel 2) ---")
    print(classification_report(y_true_xgb, y_pred_xgb_smote, target_names=['Normal (0)', 'Fraud (1)']))



--- [Baseline 2] Melatih Supervised (XGBoost) + SMOTE (Simulasi) ---
Jumlah fraud di set training (70% pertama): 1425
Menerapkan SMOTE pada data training...
Shape data training SMOTE: (144670, 5)
Melatih XGBClassifier pada data SMOTE...
Mengevaluasi model pada data tes kronologis asli...

--- Hasil XGBoost + SMOTE (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       0.99      0.99      0.99     15480
   Fraud (1)       0.63      0.58      0.60       327

    accuracy                           0.98     15807
   macro avg       0.81      0.79      0.80     15807
weighted avg       0.98      0.98      0.98     15807



In [5]:
print("\n--- [Baseline 3] Melatih AE Statis (Model Pemenang N2) ---")

# (Definisi Class StaticDataset tidak berubah)
class StaticDataset(Dataset):
    def __init__(self, df):
        # Asumsikan 'anomaly_type' ada di df, tapi kita tidak ingin itu jadi fitur
        features_df = df.drop(['Class', 'anomaly_type'], axis=1, errors='ignore')
        self.features = features_df.values
        self.labels = df['Class'].values
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# (Definisi Arsitektur Static_Autoencoder tidak berubah)
class Static_Autoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=10):
        super(Static_Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 20), nn.ReLU(),
            nn.Linear(20, embedding_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 20), nn.ReLU(),
            nn.Linear(20, n_features)
        )
    def forward(self, x):
        x = self.encoder(x); x = self.decoder(x); return x

# (DataLoaders Statis)
N_FEATURES_SIM = X_train_statis.shape[1]
train_ds_statis = StaticDataset(df_train_normal) 
val_ds_statis_mixed = StaticDataset(df_val_full) # (CAMPURAN untuk optimasi threshold)
test_ds_statis_mixed = StaticDataset(df_test_mixed)

train_loader_statis = DataLoader(train_ds_statis, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_statis_mixed = DataLoader(val_ds_statis_mixed, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_statis_mixed = DataLoader(test_ds_statis_mixed, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# (Inisialisasi & Training Loop)
model_ae_statis = Static_Autoencoder(n_features=N_FEATURES_SIM).to(device)
loss_fn_statis = nn.MSELoss()
optimizer_statis = optim.Adam(model_ae_statis.parameters(), lr=1e-3, weight_decay=1e-5)

print("Melatih AE Statis...")
for epoch in range(N_EPOCHS):
    model_ae_statis.train()
    for (data, _) in train_loader_statis:
        data = data.to(device)
        reconstructions = model_ae_statis(data)
        loss = loss_fn_statis(reconstructions, data)
        optimizer_statis.zero_grad(); loss.backward(); optimizer_statis.step()
    if (epoch+1) % 10 == 0:
      print(f"AE Statis Epoch {epoch+1}/{N_EPOCHS}")

print("Pelatihan AE Statis selesai.")

# (Optimasi Threshold untuk AE Statis)
print("Mengoptimasi Threshold AE Statis...")
model_ae_statis.eval()
val_errors_statis_mixed, val_labels_statis_mixed = [], []
with torch.no_grad():
    for (data, labels) in val_loader_statis_mixed:
        data = data.to(device)
        reconstructions = model_ae_statis(data)
        errors = torch.mean((data - reconstructions) ** 2, dim=1).cpu().numpy()
        val_errors_statis_mixed.extend(errors); val_labels_statis_mixed.extend(labels.numpy())

precision, recall, thresholds = precision_recall_curve(
    val_labels_statis_mixed, val_errors_statis_mixed
)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-10)
f1_scores = f1_scores[:-1]; precision = precision[:-1]; recall = recall[:-1]
best_f1_idx = np.argmax(f1_scores)
best_threshold_statis = thresholds[best_f1_idx]
print(f"Threshold Terbaik AE Statis: {best_threshold_statis:.6f} (F1: {f1_scores[best_f1_idx]:.4f})")

# (Evaluasi Final AE Statis)
test_errors_statis_mixed, test_labels_statis_mixed = [], []
with torch.no_grad():
    for (data, labels) in test_loader_statis_mixed:
        data = data.to(device)
        reconstructions = model_ae_statis(data)
        errors = torch.mean((data - reconstructions) ** 2, dim=1).cpu().numpy()
        test_errors_statis_mixed.extend(errors); test_labels_statis_mixed.extend(labels.numpy())

y_pred_ae_statis = [1 if e > best_threshold_statis else 0 for e in test_errors_statis_mixed]
y_true_ae_statis = test_labels_statis_mixed

print("\n--- Hasil AE Statis (Optimized) (Tabel 2) ---")
print(classification_report(y_true_ae_statis, y_pred_ae_statis, target_names=['Normal (0)', 'Fraud (1)']))



--- [Baseline 3] Melatih AE Statis (Model Pemenang N2) ---
Melatih AE Statis...
AE Statis Epoch 10/30
AE Statis Epoch 20/30
AE Statis Epoch 30/30
Pelatihan AE Statis selesai.
Mengoptimasi Threshold AE Statis...
Threshold Terbaik AE Statis: 827.455872 (F1: 0.0580)

--- Hasil AE Statis (Optimized) (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       0.99      0.06      0.12     15480
   Fraud (1)       0.02      0.98      0.04       327

    accuracy                           0.08     15807
   macro avg       0.51      0.52      0.08     15807
weighted avg       0.97      0.08      0.12     15807



In [6]:
print("\n--- [Model Utama] Melatih LSTM-AE (Simulasi) ---")

# (Definisi Class FraudDatasetSeq)
class FraudDatasetSeq(Dataset):
    def __init__(self, df, time_steps=10):
        features_df = df.drop(['Class', 'anomaly_type'], axis=1, errors='ignore')
        self.features = features_df.values
        self.labels = df['Class'].values
        self.time_steps = time_steps
    def __len__(self):
        return len(self.features) - self.time_steps + 1
    def __getitem__(self, idx):
        x = self.features[idx : (idx + self.time_steps)]
        y_label = self.labels[idx + self.time_steps - 1]
        x_tensor = torch.tensor(x, dtype=torch.float32)
        y_tensor = torch.tensor(y_label, dtype=torch.long)
        return x_tensor, y_tensor

# (Definisi Arsitektur LSTM_Autoencoder_V2)
class LSTM_Autoencoder_V2(nn.Module):
    # ... (Salin-tempel arsitektur LSTM_Autoencoder_V2 dari N2/V4) ...
    def __init__(self, n_features, time_steps, embedding_dim=16, dropout_rate=0.2):
        super(LSTM_Autoencoder_V2, self).__init__()
        self.n_features = n_features
        self.time_steps = time_steps
        self.embedding_dim = embedding_dim
        # Encoder
        self.lstm_encoder1 = nn.LSTM(
            input_size=n_features, hidden_size=64,
            batch_first=True, num_layers=1, bidirectional=True 
        )
        self.dropout_enc = nn.Dropout(dropout_rate)
        self.lstm_encoder2 = nn.LSTM(
            input_size=128, hidden_size=32,
            batch_first=True, num_layers=1, bidirectional=False
        )
        # Bottleneck
        self.bottleneck = nn.Linear(32, embedding_dim)
        # Decoder
        self.decoder_linear = nn.Linear(embedding_dim, 32)
        self.lstm_decoder1 = nn.LSTM(
            input_size=32, hidden_size=64,
            batch_first=True, num_layers=1, bidirectional=False
        )
        self.dropout_dec = nn.Dropout(dropout_rate)
        self.lstm_decoder2 = nn.LSTM(
            input_size=64, hidden_size=128,
            batch_first=True, num_layers=1, bidirectional=False
        )
        # Output layer
        self.output_layer = nn.Linear(128, n_features)

    def forward(self, x):
        x, _ = self.lstm_encoder1(x); x = self.dropout_enc(x)
        x, (hidden_n, _) = self.lstm_encoder2(x)
        x = hidden_n.squeeze(0); x = self.bottleneck(x)
        x = self.decoder_linear(x)
        x = x.unsqueeze(1).repeat(1, self.time_steps, 1)
        x, _ = self.lstm_decoder1(x); x = self.dropout_dec(x)
        x, _ = self.lstm_decoder2(x); x = self.output_layer(x)
        return x

# (DataLoaders Sekuensial)
train_ds_seq = FraudDatasetSeq(df_train_normal, time_steps=TIME_STEPS)
val_ds_seq_mixed = FraudDatasetSeq(df_val_full, time_steps=TIME_STEPS) # (CAMPURAN untuk optimasi)
test_ds_seq_mixed = FraudDatasetSeq(df_test_mixed, time_steps=TIME_STEPS)

train_loader_seq = DataLoader(train_ds_seq, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_seq_mixed = DataLoader(val_ds_seq_mixed, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_seq_mixed = DataLoader(test_ds_seq_mixed, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# (Inisialisasi & Training Loop)
model_lstm_ae = LSTM_Autoencoder_V2(
    n_features=N_FEATURES_SIM, 
    time_steps=TIME_STEPS
).to(device)
loss_fn_lstm = nn.MSELoss()
optimizer_lstm = optim.Adam(model_lstm_ae.parameters(), lr=1e-3, weight_decay=1e-5)

print("Melatih Model Utama (LSTM-AE)...")
for epoch in range(N_EPOCHS_LSTM): # (Latih lebih lama)
    model_lstm_ae.train()
    for (seq_data, _) in train_loader_seq:
        seq_data = seq_data.to(device)
        reconstructions = model_lstm_ae(seq_data)
        loss = loss_fn_lstm(reconstructions, seq_data)
        optimizer_lstm.zero_grad(); loss.backward(); optimizer_lstm.step()
    if (epoch+1) % 10 == 0 or epoch == 0:
      print(f"LSTM-AE Epoch {epoch+1}/{N_EPOCHS_LSTM}")
print("Pelatihan LSTM-AE selesai.")

# (Optimasi Threshold untuk LSTM-AE)
print("Mengoptimasi Threshold LSTM-AE...")
model_lstm_ae.eval()
val_errors_lstm_mixed, val_labels_lstm_mixed = [], []
with torch.no_grad():
    for (data, labels) in val_loader_seq_mixed:
        data = data.to(device)
        reconstructions = model_lstm_ae(data)
        errors = torch.mean((data - reconstructions) ** 2, dim=(1,2)).cpu().numpy()
        val_errors_lstm_mixed.extend(errors); val_labels_lstm_mixed.extend(labels.numpy())

precision_l, recall_l, thresholds_l = precision_recall_curve(
    val_labels_lstm_mixed, val_errors_lstm_mixed
)
f1_scores_l = (2 * precision_l * recall_l) / (precision_l + recall_l + 1e-10)
f1_scores_l = f1_scores_l[:-1];
best_f1_idx_l = np.argmax(f1_scores_l)
best_threshold_lstm = thresholds_l[best_f1_idx_l]
print(f"Threshold Terbaik LSTM-AE: {best_threshold_lstm:.6f} (F1: {f1_scores_l[best_f1_idx_l]:.4f})")

# (Evaluasi Final LSTM-AE)
test_errors_lstm_mixed, test_labels_lstm_mixed = [], []
with torch.no_grad():
    for (data, labels) in test_loader_seq_mixed:
        data = data.to(device)
        reconstructions = model_lstm_ae(data)
        errors = torch.mean((data - reconstructions) ** 2, dim=(1,2)).cpu().numpy()
        test_errors_lstm_mixed.extend(errors); test_labels_lstm_mixed.extend(labels.numpy())

y_pred_lstm = [1 if e > best_threshold_lstm else 0 for e in test_errors_lstm_mixed]
y_true_lstm = test_labels_lstm_mixed

print("\n--- Hasil Model Utama (LSTM-AE Optimized) (Tabel 2) ---")
print(classification_report(y_true_lstm, y_pred_lstm, target_names=['Normal (0)', 'Fraud (1)']))
print("HIPOTESIS: F1-Score ini seharusnya mengalahkan F1-Score AE Statis.")



--- [Model Utama] Melatih LSTM-AE (Simulasi) ---
Melatih Model Utama (LSTM-AE)...
LSTM-AE Epoch 1/40
LSTM-AE Epoch 10/40
LSTM-AE Epoch 20/40
LSTM-AE Epoch 30/40
LSTM-AE Epoch 40/40
Pelatihan LSTM-AE selesai.
Mengoptimasi Threshold LSTM-AE...
Threshold Terbaik LSTM-AE: 893930688.000000 (F1: 0.0434)

--- Hasil Model Utama (LSTM-AE Optimized) (Tabel 2) ---
              precision    recall  f1-score   support

  Normal (0)       0.00      0.00      0.00     15472
   Fraud (1)       0.02      1.00      0.04       326

    accuracy                           0.02     15798
   macro avg       0.01      0.50      0.02     15798
weighted avg       0.00      0.02      0.00     15798

HIPOTESIS: F1-Score ini seharusnya mengalahkan F1-Score AE Statis.


In [7]:
print("\n--- [Analisis Final] Menguji Performa pada Anomali Sekuensial (Tabel 4) ---")

# Kita perlu mencocokkan prediksi dengan tipe anomali
# (Ingat: dataset sekuensial lebih pendek TIME_STEPS-1)
offset = TIME_STEPS - 1
df_test_labels_seq = df_test_mixed.iloc[offset:].copy()

# Buat dataframe hasil
df_results = pd.DataFrame({
    'y_true': df_test_labels_seq['Class'],
    'anomaly_type': df_test_labels_seq['anomaly_type'],
    'pred_AE_Statis': y_pred_ae_statis[offset:], # (Sesuaikan panjang)
    'pred_LSTM_AE': y_pred_lstm 
})

# Pisahkan hanya data fraud
df_fraud_results = df_results[df_results['y_true'] == 1].copy()

# 1. Analisis Anomali Statis (Tipe 1 & 2)
df_fraud_statis = df_fraud_results[df_fraud_results['anomaly_type'].isin([1, 2])]
recall_ae_statis = recall_score(df_fraud_statis['y_true'], df_fraud_statis['pred_AE_Statis'])
recall_lstm_statis = recall_score(df_fraud_statis['y_true'], df_fraud_statis['pred_LSTM_AE'])

print("\n--- Hasil pada Anomali Statis (Tipe 1 & 2) ---")
print(f"Recall AE Statis (Model Pemenang N2): {recall_ae_statis:.4f}")
print(f"Recall LSTM-AE: {recall_lstm_statis:.4f}")

# 2. Analisis Anomali Sekuensial (Tipe 3)
df_fraud_seq = df_fraud_results[df_fraud_results['anomaly_type'] == 3]
recall_ae_seq = recall_score(df_fraud_seq['y_true'], df_fraud_seq['pred_AE_Statis'])
recall_lstm_seq = recall_score(df_fraud_seq['y_true'], df_fraud_seq['pred_LSTM_AE'])

print("\n--- Hasil pada AnomALI SEKUENTIAL (Tipe 3) ---")
print(f"Recall AE Statis: {recall_ae_seq:.4f}  <-- HARAPAN: RENDAH")
print(f"Recall LSTM-AE: {recall_lstm_seq:.4f}  <-- HARAPAN: TINGGI")

# (Simpan ke CSV untuk Tabel 4 di jurnal)
tabel_4_data = {
    'Model': ['AE Statis (Optimized)', 'LSTM-AE (Optimized)'],
    'Recall (Anomali Statis, Tipe 1&2)': [recall_ae_statis, recall_lstm_statis],
    'Recall (Anomali Sekuensial, Tipe 3)': [recall_ae_seq, recall_lstm_seq]
}
df_tabel_4 = pd.DataFrame(tabel_4_data)
df_tabel_4.to_csv(os.path.join(MODEL_PATH, 'tabel_4_analisis_subtipe.csv'), index=False)
print("\nData Tabel 4 disimpan ke CSV.")
print(df_tabel_4)

print("\nNotebook 3 Selesai!")



--- [Analisis Final] Menguji Performa pada Anomali Sekuensial (Tabel 4) ---

--- Hasil pada Anomali Statis (Tipe 1 & 2) ---
Recall AE Statis (Model Pemenang N2): 0.9884
Recall LSTM-AE: 1.0000

--- Hasil pada AnomALI SEKUENTIAL (Tipe 3) ---
Recall AE Statis: 0.9792  <-- HARAPAN: RENDAH
Recall LSTM-AE: 1.0000  <-- HARAPAN: TINGGI

Data Tabel 4 disimpan ke CSV.
                   Model  Recall (Anomali Statis, Tipe 1&2)  \
0  AE Statis (Optimized)                           0.988372   
1    LSTM-AE (Optimized)                           1.000000   

   Recall (Anomali Sekuensial, Tipe 3)  
0                             0.979167  
1                             1.000000  

Notebook 3 Selesai!
